# 07 Intervention API

This notebook audits the interactive intervention surface: finding sites, forking a trace, attaching hooks, `do`, `set`, `replay`, and `rerun`. An intervention changes a recorded activation and then propagates that change through the saved graph or a fresh model run.

The model has a single ReLU site with a deliberately nonzero activation. We zero that site and verify that both the targeted tensor and the final output change.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(31)


class InterventionNet(nn.Module):
    """Tiny model with one nonzero ReLU intervention site."""

    def __init__(self) -> None:
        """Initialize layers with deterministic nonzero intervention behavior."""

        super().__init__()
        self.stem = nn.Linear(3, 3)
        self.head = nn.Linear(3, 1)
        with torch.no_grad():
            self.stem.weight.copy_(
                torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.5, 0.5, 1.0]])
            )
            self.stem.bias.copy_(torch.tensor([0.2, 0.3, 0.1]))
            self.head.weight.copy_(torch.tensor([[1.0, -2.0, 0.5]]))
            self.head.bias.copy_(torch.tensor([0.05]))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the model.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            One-column output.
        """

        hidden = torch.relu(self.stem(x))
        return self.head(hidden)


def relu_record(log: tl.Trace) -> tl.Layer:
    """Return the ReLU layer from an intervention trace."""

    return next(layer for layer in log.layer_list if layer.func_name == "relu")


def print_intervention_delta(name: str, log: tl.Trace) -> None:
    """Print targeted activation and final-output deltas for one intervention."""

    clean_relu = relu_layer.out.detach()
    edited_relu = relu_record(log).out.detach()
    clean_out = clean[output_label].out.detach()
    edited_out = log[output_label].out.detach()
    print(f"{name} relu sum: {edited_relu.sum().item():.6f}")
    print(f"{name} relu max delta: {(edited_relu - clean_relu).abs().max().item():.6f}")
    print(f"{name} output: {edited_out.flatten().tolist()}")
    print(f"{name} output max delta: {(edited_out - clean_out).abs().max().item():.6f}")
    print(f"{name} output changed: {not torch.allclose(clean_out, edited_out)}")


model = InterventionNet().eval()
x = torch.tensor([[1.0, 0.5, 0.25]])
clean = tl.trace(model, x, intervention_ready=True)
relu_layer = relu_record(clean)
output_label = clean.output_layers[0]
print(f"clean output: {clean[output_label].out.flatten().tolist()}")
print(f"relu layer: {relu_layer.layer_label}")
print(f"relu tensor: {relu_layer.out.detach().flatten().tolist()}")
print(f"relu sum: {relu_layer.out.sum().item():.6f}")
if relu_layer.out.abs().sum().item() == 0.0:
    raise RuntimeError("intervention example requires a nonzero ReLU activation")

`find_sites` resolves selectors against a trace. In this build it is a `Trace` method; selectors such as `tl.func("relu")` are top-level helpers.

In [ ]:
sites = clean.find_sites(tl.func("relu"))
print(sites)
print(f"top-level find_sites exists: {hasattr(tl, 'find_sites')}")

Forking creates a separate trace object so the clean trace remains unchanged while we experiment.

In [ ]:
forked = clean.fork(name="zero_relu")
print(f"fork is clean: {forked is clean}")
print(f"clean state: {clean.state}")
print(f"fork state: {forked.state}")

`attach_hooks(site, hook)` stores a sticky hook recipe. `tl.zero_ablate()` is a helper that replaces the matched activation with zeros. `replay()` then propagates the changed value through the saved graph.

In [ ]:
forked.attach_hooks(tl.func("relu"), tl.zero_ablate(), confirm_mutation=True)
print(f"after attach: {forked.state}")
forked.replay(strict=False)
print_intervention_delta("attach_hooks/replay", forked)

`do` is the convenient dispatcher. With `engine="replay"` it attaches the intervention and replays the affected cone in one call.

In [ ]:
do_log = clean.fork(name="do_zero_relu")
do_log.do(
    tl.func("relu"),
    tl.zero_ablate(),
    engine="replay",
    confirm_mutation=True,
    strict=False,
)
print(f"do state: {do_log.state}")
print_intervention_delta("do", do_log)

`set` records a direct replacement recipe. It marks the trace stale until `replay()` propagates downstream values.

In [ ]:
set_log = clean.fork(name="set_relu")
replacement = torch.zeros_like(relu_layer.out)
set_log.set(tl.func("relu"), replacement, confirm_mutation=True)
print(f"after set: {set_log.state}")
set_log.replay(strict=False)
print(f"after replay: {set_log.state}")
print_intervention_delta("set/replay", set_log)

`rerun` re-executes the model with the active intervention recipe. Use it when replay is not enough, such as with dynamic control flow or when you want a fresh forward pass.

In [ ]:
rerun_log = clean.fork(name="rerun_zero_relu")
rerun_log.attach_hooks(tl.func("relu"), tl.zero_ablate(), confirm_mutation=True)
rerun_log.rerun(model, x, strict=False)
print(f"rerun state: {rerun_log.state}")
print_intervention_delta("rerun", rerun_log)

> NOTE: The glossary describes top-level `find_sites`; the current executable API exposes `Trace.find_sites(...)`. The top-level selector helpers, including `tl.func`, `tl.label`, `tl.where`, and `tl.in_module`, are available.

## 🔧 Sandbox

Mini-experiment: change `sandbox_engine`, `sandbox_scale`, and `sandbox_selector`. Expected observation: stronger scaling changes the ReLU replacement and the final output by a visible amount.

In [ ]:
sandbox_engine = "replay"
sandbox_scale = 0.25
sandbox_selector = tl.func("relu")
sandbox_replacement = relu_layer.out * sandbox_scale
sandbox_log = clean.fork(name="sandbox")
sandbox_log.do(
    sandbox_selector,
    sandbox_replacement,
    model=model,
    x=x,
    engine=sandbox_engine,
    confirm_mutation=True,
    strict=False,
)
print(f"engine: {sandbox_engine}")
print(f"scale: 1.0 -> {sandbox_scale}")
print_intervention_delta("sandbox", sandbox_log)